# SuperVet — Módulo de IA (Sprint 4)
**Abordagem híbrida:** motor de regras (decide e justifica) + LLM local via **Ollama** (só redige o texto).

| Camada do documento (Sprint 3) | Onde está neste notebook |
|---|---|
| Dados (TB_PET, TB_PRONTUARIO, TB_COLEIRA…) | Seção 2 — dados simulados com os mesmos nomes de tabela/coluna |
| Montagem de contexto sem dados pessoais | Seção 3 — `PetContextBuilder` |
| Motor de regras / score de prioridade | Seção 4 — `calcular_score` |
| IA Generativa (LLM) | Seção 5 — notificação, resumo pré-consulta e chat do tutor |
| Módulo Orquestrador | Seção 6 — `analisar_pet`, `fila_clinica` |
| Testes e validação | Seção 7 |
| Integração com a API Java | Seção 9 — FastAPI + túnel público |

**Como rodar:** `Ambiente de execução → Alterar tipo → GPU T4` e depois `Executar tudo`.
Sem GPU, troque `MODELO` por `llama3.2:3b` na Seção 1.

## 1. Ambiente: Ollama no Colab

In [ ]:
# Instala o Ollama (o instalador atual precisa do zstd)
!nvidia-smi -L || echo "Sem GPU detectada: use um modelo menor (ex.: llama3.2:3b)"
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Configuração (única fonte de verdade para o resto do notebook)
import requests

MODELO = "qwen2.5:7b"          # alternativas: "llama3.1:8b" | "llama3.2:3b" (sem GPU)
OLLAMA_URL = "http://127.0.0.1:11434"

def ollama_ativo():
    try:
        return requests.get(OLLAMA_URL, timeout=2).status_code == 200
    except Exception:
        return False

In [ ]:
# Sobe o servidor em segundo plano e baixa o modelo
import subprocess, time

if not ollama_ativo():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(30):
        if ollama_ativo():
            break
        time.sleep(1)
print("Ollama ativo:", ollama_ativo())

!ollama pull {MODELO}

In [ ]:
# Teste de fumaça: o modelo responde em português?
r = requests.post(f"{OLLAMA_URL}/api/chat", json={
    "model": MODELO, "stream": False,
    "messages": [{"role": "user", "content": "Responda só: OK, SuperVet pronto."}]}, timeout=300)
print(r.json()["message"]["content"])

## 2. Dados de teste (simulam o banco Oracle)
Mesmos nomes de tabelas e colunas do documento do Sprint 3. Todas as datas são **relativas a hoje**, então os cenários continuam válidos em qualquer dia.
São 4 pets em situações diferentes: saudável, vacina/retorno atrasados, perda de peso com sintomas recorrentes e anomalia cardíaca com alerta da coleira.

In [ ]:
import random, json, re, unicodedata
from collections import Counter
from dataclasses import dataclass, asdict
from datetime import datetime, date, timedelta

AGORA = datetime.now().replace(microsecond=0)
HOJE = AGORA.date()
def passado(dias): return (HOJE - timedelta(days=dias)).isoformat()
def futuro(dias):  return (HOJE + timedelta(days=dias)).isoformat()

TB_ESPECIE = {1: {"NOME": "Cachorro"}, 2: {"NOME": "Gato"}}
TB_RACA = {
    1: {"NOME": "Labrador",      "PORTE": "GRANDE",  "ID_ESPECIE": 1},
    2: {"NOME": "Shih Tzu",      "PORTE": "PEQUENO", "ID_ESPECIE": 1},
    3: {"NOME": "SRD",           "PORTE": "MEDIO",   "ID_ESPECIE": 2},
    4: {"NOME": "Pastor Alemão", "PORTE": "GRANDE",  "ID_ESPECIE": 1},
}
# Dados pessoais fictícios: existem só para provar que NÃO saem do backend
TB_TUTOR = {i: {"NOME": f"Tutor Exemplo {i}", "CPF": f"000.000.00{i}-0{i}",
                "EMAIL": f"tutor{i}@exemplo.com", "TELEFONE": f"(11) 90000-000{i}"} for i in range(1, 5)}
TB_PET = {
    1: {"NOME": "Thor", "SEXO": "M", "DATA_NASCIMENTO": passado(5 * 365),  "ID_RACA": 1, "ID_TUTOR": 1},
    2: {"NOME": "Mel",  "SEXO": "F", "DATA_NASCIMENTO": passado(9 * 365),  "ID_RACA": 2, "ID_TUTOR": 2},
    3: {"NOME": "Luna", "SEXO": "F", "DATA_NASCIMENTO": passado(12 * 365), "ID_RACA": 3, "ID_TUTOR": 3},
    4: {"NOME": "Rex",  "SEXO": "M", "DATA_NASCIMENTO": passado(7 * 365),  "ID_RACA": 4, "ID_TUTOR": 4},
}

def _pront(id_pet, dias, diag, sintomas, peso, temp, exames, retorno):
    return {"ID_PET": id_pet, "DATA": passado(dias), "DIAGNOSTICO": diag, "SINTOMAS": sintomas,
            "PESO": peso, "TEMPERATURA": temp, "EXAMES_SOLICITADOS": exames, "DATA_RETORNO": retorno}

TB_PRONTUARIO = [
    _pront(1, 200, "Check-up anual", "", 31.8, 38.4, "", None),
    _pront(1, 45,  "Check-up de rotina, sem alterações", "", 32.0, 38.5, "", futuro(45)),
    _pront(2, 180, "Avaliação geral", "", 7.1, 38.6, "", None),
    _pront(2, 50,  "Tártaro dentário grau II", "mau hálito", 7.2, 38.7, "Limpeza dentária (agendar)", passado(20)),
    _pront(3, 70,  "Episódio de vômito isolado", "vômito", 4.8, 38.4, "", None),
    _pront(3, 24,  "Doença renal crônica em investigação", "vômito, apatia", 4.2, 38.3, "ureia, creatinina", passado(10)),
    _pront(4, 120, "Avaliação geral", "", 36.0, 38.6, "", None),
    _pront(4, 10,  "Tosse persistente em investigação", "tosse, cansaço", 35.5, 38.8, "raio-x de tórax, ecocardiograma", passado(3)),
]

def _cons(id_pet, dias, status): return {"ID_PET": id_pet, "DATA": passado(dias), "STATUS": status, "MODALIDADE": "PRESENCIAL"}
TB_CONSULTA = [
    _cons(1, 45, "REALIZADA"), _cons(1, 200, "REALIZADA"),
    _cons(2, 50, "REALIZADA"), _cons(2, 180, "REALIZADA"),
    _cons(3, 24, "REALIZADA"), _cons(3, 70, "REALIZADA"),
    _cons(4, 10, "REALIZADA"), _cons(4, 120, "REALIZADA"), _cons(4, 3, "FALTOU"),
]

def _hist(id_pet, dias, evento, detalhes): return {"ID_PET": id_pet, "EVENTO": evento, "DATA_EVENTO": passado(dias), "DETALHES": detalhes}
TB_HISTORICO_SAUDE = [
    _hist(1, 120, "VACINA", "V10 e antirrábica"),
    _hist(2, 420, "VACINA", "V10"),
    _hist(3, 200, "VACINA", "V4 felina"),
    _hist(4, 100, "VACINA", "V10"),
]

TB_PRESCRICAO = [
    {"ID_PET": 2, "TIPO": "DIETA", "DESCRICAO": "Ração para saúde bucal", "POSOLOGIA": "Conforme embalagem", "ATIVA": True},
    {"ID_PET": 3, "TIPO": "DIETA", "DESCRICAO": "Ração renal", "POSOLOGIA": "Conforme orientação do veterinário", "ATIVA": True},
    {"ID_PET": 3, "TIPO": "MEDICAMENTO", "DESCRICAO": "Protetor gástrico", "POSOLOGIA": "Conforme receita do veterinário", "ATIVA": True},
    {"ID_PET": 4, "TIPO": "RESTRICAO", "DESCRICAO": "Evitar exercício intenso até a reavaliação", "POSOLOGIA": "", "ATIVA": True},
]

def _gerar_coleira(id_pet, normal, anomala=None, n_anomalas=0, n_alertas=0, seed=0):
    # Uma leitura a cada 30 min nas últimas 24h. As últimas `n_anomalas` ficam fora da faixa.
    rnd = random.Random(seed)
    leituras = []
    for i in range(48, 0, -1):
        leituras.append({"ID_PET": id_pet, "DATA_HORA": (AGORA - timedelta(minutes=30 * i)).isoformat(),
                         "LATITUDE": -23.55 + rnd.uniform(-0.001, 0.001), "LONGITUDE": -46.63 + rnd.uniform(-0.001, 0.001),
                         "BPM": rnd.randint(*normal), "ALERTA": None})
    for k in range(1, n_anomalas + 1):
        leituras[-k]["BPM"] = rnd.randint(*anomala)
    for k in range(1, n_alertas + 1):
        leituras[-k]["ALERTA"] = "TAQUICARDIA"
    return leituras

TB_COLEIRA = (
    _gerar_coleira(1, (65, 95), seed=1) +
    _gerar_coleira(2, (105, 135), seed=2) +
    _gerar_coleira(3, (150, 190), seed=3) +
    _gerar_coleira(4, (70, 95), anomala=(115, 140), n_anomalas=5, n_alertas=2, seed=4)
)
print(f"{len(TB_PET)} pets | {len(TB_PRONTUARIO)} prontuários | {len(TB_COLEIRA)} leituras de coleira")

## 3. `PetContextBuilder` — minimização de dados (LGPD)
Junta as tabelas de um pet em um único JSON e **descarta** CPF, e-mail, telefone, nome do tutor e GPS.
Esse é o contrato: o backend Java deve montar exatamente este JSON (Seção 9).

In [ ]:
def _norm(s):
    s = unicodedata.normalize("NFD", str(s or ""))
    return "".join(c for c in s if unicodedata.category(c) != "Mn").lower().strip()

class PetContextBuilder:
    def montar(self, id_pet, agora=AGORA):
        pet = TB_PET[id_pet]
        raca = TB_RACA[pet["ID_RACA"]]
        especie = TB_ESPECIE[raca["ID_ESPECIE"]]["NOME"]
        nasc = date.fromisoformat(pet["DATA_NASCIMENTO"])
        corte = (agora - timedelta(hours=24)).isoformat()
        return {
            "id_pet": id_pet,
            "referencia_em": agora.isoformat(),
            "pet": {"nome": pet["NOME"], "sexo": pet["SEXO"], "especie": especie, "raca": raca["NOME"],
                    "porte": raca["PORTE"], "idade_anos": (agora.date() - nasc).days // 365},
            "prontuarios": sorted(
                [{"data": p["DATA"], "diagnostico": p["DIAGNOSTICO"], "sintomas": p["SINTOMAS"], "peso_kg": p["PESO"],
                  "temperatura_c": p["TEMPERATURA"], "exames_solicitados": p["EXAMES_SOLICITADOS"],
                  "data_retorno": p["DATA_RETORNO"]} for p in TB_PRONTUARIO if p["ID_PET"] == id_pet],
                key=lambda x: x["data"]),
            "consultas": [{"data": c["DATA"], "status": c["STATUS"], "modalidade": c["MODALIDADE"]}
                          for c in TB_CONSULTA if c["ID_PET"] == id_pet],
            "historico_saude": sorted(
                [{"evento": h["EVENTO"], "data": h["DATA_EVENTO"], "detalhes": h["DETALHES"]}
                 for h in TB_HISTORICO_SAUDE if h["ID_PET"] == id_pet], key=lambda x: x["data"]),
            "prescricoes": [{"tipo": r["TIPO"], "descricao": r["DESCRICAO"], "posologia": r["POSOLOGIA"], "ativa": r["ATIVA"]}
                            for r in TB_PRESCRICAO if r["ID_PET"] == id_pet],
            # Sem LATITUDE/LONGITUDE: a localização não é necessária para a IA
            "coleira_24h": [{"datahora": c["DATA_HORA"], "bpm": c["BPM"], "alerta": c["ALERTA"]}
                            for c in TB_COLEIRA if c["ID_PET"] == id_pet and c["DATA_HORA"] >= corte],
        }

builder = PetContextBuilder()
CONTEXTOS = [builder.montar(i) for i in TB_PET]
print(json.dumps({k: (v if k != "coleira_24h" else f"[{len(v)} leituras]") for k, v in CONTEXTOS[3].items()},
                 ensure_ascii=False, indent=2)[:1800])

## 4. Motor de regras — score de prioridade (determinístico e auditável)
Cada regra devolve **pontos + motivo**. O score é a soma (máx. 100). Os limiares ficam em um dicionário para o veterinário poder revisar.

> ⚠️ Faixas de BPM e protocolo vacinal são **valores de referência ilustrativos** e devem ser validados por um veterinário. Como a coleira não informa o nível de atividade, leituras durante exercício podem gerar falso positivo (limitação a documentar).

In [ ]:
LIMIARES = {
    "retorno_atraso_pts":   [(14, 15), (60, 25), (10**6, 30)],   # (até N dias de atraso, pontos)
    "protocolo_vacinal_dias": 365,                                # placeholder até existir a tabela de protocolos (Fase 2)
    "vacina_pts": 20,
    "peso_pct": [(10, 20), (5, 10)],                              # (variação >= %, pontos)
    "sintomas_recentes_dias": 30, "sintomas_recentes_pts": 10,
    "sintoma_recorrente_janela_dias": 180, "sintoma_recorrente_pts": 10,
    "bpm_persistente_min_leituras": 3, "bpm_persistente_pts": 25, "bpm_isolado_pts": 10,
    "alerta_coleira_pts": 20,
    "consulta_perdida_dias": 90, "consulta_perdida_pts": 10,
    "niveis": [(75, "CRITICO"), (50, "ALTO"), (25, "MEDIO"), (0, "BAIXO")],
}

def faixa_bpm(especie, porte):
    if _norm(especie) == "gato":
        return (140, 220)
    return {"pequeno": (100, 140), "medio": (80, 120), "grande": (60, 100)}.get(_norm(porte), (60, 140))

@dataclass
class Achado:
    codigo: str
    pontos: int
    motivo: str

def _d(s): return date.fromisoformat(str(s)[:10])
def _br(d): return d.strftime("%d/%m/%Y")

def regra_retorno(ctx, hoje):
    if not ctx["prontuarios"]: return []
    ult = max(ctx["prontuarios"], key=lambda p: p["data"])
    if not ult.get("data_retorno"): return []
    ret = _d(ult["data_retorno"])
    if ret >= hoje: return []
    if any(_norm(c["status"]) in {"realizada", "concluida"} and _d(c["data"]) > _d(ult["data"]) for c in ctx["consultas"]):
        return []
    atraso = (hoje - ret).days
    pts = next(p for lim, p in LIMIARES["retorno_atraso_pts"] if atraso <= lim)
    return [Achado("RETORNO_ATRASADO", pts, f"O retorno previsto para {_br(ret)} está atrasado há {atraso} dias.")]

def regra_vacina(ctx, hoje):
    vac = [e for e in ctx["historico_saude"] if "vacina" in _norm(e["evento"])]
    if not vac:
        return [Achado("VACINA_SEM_REGISTRO", LIMIARES["vacina_pts"], "Não há registro de vacinação no histórico.")]
    ult = max(_d(e["data"]) for e in vac)
    dias = (hoje - ult).days
    if dias > LIMIARES["protocolo_vacinal_dias"]:
        return [Achado("VACINA_ATRASADA", LIMIARES["vacina_pts"],
                       f"A última vacinação foi em {_br(ult)} (há {dias} dias), acima do intervalo de referência de {LIMIARES['protocolo_vacinal_dias']} dias.")]
    return []

def regra_peso(ctx, hoje):
    pesos = [p for p in sorted(ctx["prontuarios"], key=lambda p: p["data"]) if p.get("peso_kg")]
    if len(pesos) < 2: return []
    ant, atual = pesos[-2]["peso_kg"], pesos[-1]["peso_kg"]
    var = (atual - ant) / ant * 100
    for lim, pts in LIMIARES["peso_pct"]:
        if abs(var) >= lim:
            verbo = "caiu" if var < 0 else "subiu"
            return [Achado("VARIACAO_PESO", pts, f"O peso {verbo} {abs(var):.1f}% entre os dois últimos prontuários ({ant:.1f} kg → {atual:.1f} kg).")]
    return []

def _tokens(s):
    return [t.strip() for t in re.split(r"[,;]", (s or "").lower()) if t.strip()]

def regra_sintomas(ctx, hoje):
    achados = []
    rec = [p for p in ctx["prontuarios"] if p.get("sintomas") and (hoje - _d(p["data"])).days <= LIMIARES["sintomas_recentes_dias"]]
    if rec:
        lista = ", ".join(sorted({t for p in rec for t in _tokens(p["sintomas"])}))
        achados.append(Achado("SINTOMAS_RECENTES", LIMIARES["sintomas_recentes_pts"],
                              f"Sintomas registrados nos últimos {LIMIARES['sintomas_recentes_dias']} dias: {lista}."))
    cont, exibicao = Counter(), {}
    for p in ctx["prontuarios"]:
        if (hoje - _d(p["data"])).days <= LIMIARES["sintoma_recorrente_janela_dias"]:
            for t in set(_tokens(p["sintomas"])):
                cont[_norm(t)] += 1
                exibicao[_norm(t)] = t
    recorrentes = [exibicao[k] for k, c in cont.items() if c >= 2]
    if recorrentes:
        achados.append(Achado("SINTOMA_RECORRENTE", LIMIARES["sintoma_recorrente_pts"],
                              f"Sintoma recorrente em mais de um prontuário: {', '.join(recorrentes)}."))
    return achados

def regra_bpm(ctx, hoje):
    leituras = [l["bpm"] for l in ctx["coleira_24h"] if l.get("bpm") is not None]
    lo, hi = faixa_bpm(ctx["pet"]["especie"], ctx["pet"]["porte"])
    fora = [b for b in leituras if not (lo <= b <= hi)]
    if not fora: return []
    persistente = len(fora) >= LIMIARES["bpm_persistente_min_leituras"]
    pts = LIMIARES["bpm_persistente_pts"] if persistente else LIMIARES["bpm_isolado_pts"]
    tipo = "BPM_PERSISTENTE" if persistente else "BPM_ISOLADO"
    return [Achado(tipo, pts, f"{len(fora)} leitura(s) de batimento fora da faixa de referência ({lo}–{hi} bpm) nas últimas 24h, com valores entre {min(fora)} e {max(fora)} bpm.")]

def regra_alerta_coleira(ctx, hoje):
    alertas = [l["alerta"] for l in ctx["coleira_24h"] if l.get("alerta")]
    if not alertas: return []
    return [Achado("ALERTA_COLEIRA", LIMIARES["alerta_coleira_pts"],
                   f"A coleira emitiu {len(alertas)} alerta(s) nas últimas 24h (tipo: {', '.join(sorted(set(alertas))).lower()}).")]

def regra_consulta_perdida(ctx, hoje):
    perdidas = [c for c in ctx["consultas"] if _norm(c["status"]) in {"faltou", "nao_compareceu"}
                and (hoje - _d(c["data"])).days <= LIMIARES["consulta_perdida_dias"]]
    if not perdidas: return []
    ult = max(_d(c["data"]) for c in perdidas)
    return [Achado("CONSULTA_PERDIDA", LIMIARES["consulta_perdida_pts"], f"O tutor não compareceu à consulta de {_br(ult)}.")]

REGRAS = [regra_retorno, regra_vacina, regra_peso, regra_sintomas, regra_bpm, regra_alerta_coleira, regra_consulta_perdida]

def nivel_de(score):
    return next(n for lim, n in LIMIARES["niveis"] if score >= lim)

def calcular_score(ctx):
    hoje = datetime.fromisoformat(ctx["referencia_em"]).date()
    achados = [a for regra in REGRAS for a in regra(ctx, hoje)]
    achados.sort(key=lambda a: -a.pontos)
    score = min(100, sum(a.pontos for a in achados))
    return {"score": score, "nivel": nivel_de(score), "achados": [asdict(a) for a in achados]}

for c in CONTEXTOS:
    s = calcular_score(c)
    print(f"{c['pet']['nome']:5} score={s['score']:3} {s['nivel']:8} {[a['codigo'] for a in s['achados']]}")

## 5. Camada de IA Generativa (Ollama)
O LLM **não decide**: recebe o score e os motivos já calculados e só escreve o texto. Proteções:
1. *System prompt* restritivo (só usa os dados fornecidos, não diagnostica, não dá dose).
2. `temperature` baixa.
3. **Guarda de emergência** no chat: palavras-chave de urgência nem chegam ao LLM, a resposta é fixa.
4. **Fallback** determinístico se o Ollama estiver fora do ar.
5. Aviso de apoio à decisão anexado pelo código, não pelo modelo.

In [ ]:
SYSTEM_BASE = """Você é o assistente de IA da plataforma SuperVet, usada por tutores e clínicas veterinárias.
Regras obrigatórias:
1. Responda sempre em português do Brasil.
2. Use SOMENTE os dados fornecidos no JSON. Se uma informação não estiver nos dados, diga que não há registro.
3. Você NÃO diagnostica, NÃO prescreve e NÃO sugere doses ou troca de medicamentos. Você é apoio à decisão; quem confirma diagnóstico e tratamento é o veterinário.
4. Não invente números, datas, nomes de medicamentos ou exames.
5. O nível e os motivos de prioridade já foram calculados pelo sistema: não recalcule nem contradiga."""

TAREFA_NOTIFICACAO = """Escreva uma notificação curta para o TUTOR do pet (máximo 3 frases), em tom acolhedor e linguagem simples, sem termos técnicos. Cite o motivo mais importante da lista 'motivos' e sugira UMA ação prática (por exemplo, agendar o retorno). Se não houver motivos, diga que está tudo em dia. Não use listas nem títulos."""

TAREFA_RESUMO = """Gere um resumo pré-consulta para o VETERINÁRIO, em até 180 palavras, com exatamente estas seções em Markdown:
**Perfil**
**O que mudou desde o último atendimento**
**Pontos de atenção (motor de regras)**
**Tratamentos em curso**
**Itens a verificar na consulta**
Em 'Itens a verificar', sugira apenas checagens e perguntas de anamnese, sem diagnosticar."""

TAREFA_CHAT = """Responda à dúvida do TUTOR em até 120 palavras, em linguagem simples. Se a resposta depender de avaliação clínica, oriente a falar com o veterinário. Se a dúvida não puder ser respondida com os dados fornecidos, diga isso."""

DISCLAIMER = "\n\n_Apoio informativo gerado por IA. Não substitui a avaliação do veterinário, que confirma diagnósticos e tratamentos._"

EMERGENCIAS = [r"convuls", r"envenen", r"veneno", r"sangr", r"hemorrag", r"nao (esta )?respir", r"falta de ar",
               r"engasg", r"desmai", r"atropel", r"intoxic", r"parou de respirar", r"nao acorda",
               r"\bchocolate\b", r"\buvas?\b", r"barriga (muito )?(inchada|dura)"]

def chamar_llm(system, user, temperature=0.2, max_tokens=400):
    r = requests.post(f"{OLLAMA_URL}/api/chat", timeout=300, json={
        "model": MODELO, "stream": False, "keep_alive": "30m",
        "options": {"temperature": temperature, "num_predict": max_tokens},
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]})
    r.raise_for_status()
    return r.json()["message"]["content"].strip()

def resumo_coleira(ctx):
    b = [x["bpm"] for x in ctx["coleira_24h"] if x.get("bpm") is not None]
    if not b: return {"leituras": 0}
    return {"leituras": len(b), "bpm_min": min(b), "bpm_medio": round(sum(b) / len(b)), "bpm_max": max(b),
            "alertas": sum(1 for x in ctx["coleira_24h"] if x.get("alerta"))}

def contexto_para_llm(ctx, score):
    # Versão compacta: nunca manda as 48 leituras cruas, só o resumo
    return {"pet": ctx["pet"],
            "prioridade": {"score": score["score"], "nivel": score["nivel"], "motivos": [a["motivo"] for a in score["achados"]]},
            "ultimos_prontuarios": ctx["prontuarios"][-3:],
            "historico_saude": ctx["historico_saude"][-5:],
            "prescricoes_ativas": [p for p in ctx["prescricoes"] if p.get("ativa", True)],
            "coleira_24h_resumo": resumo_coleira(ctx)}

def _prompt(tarefa, ctx, score, extra=""):
    dados = json.dumps(contexto_para_llm(ctx, score), ensure_ascii=False)
    return f"{tarefa}\n\nDADOS (JSON):\n{dados}\n{extra}"

# ── Fallbacks determinísticos (usados se o LLM falhar) ──
def fallback_notificacao(ctx, score):
    nome = ctx["pet"]["nome"]
    if not score["achados"]:
        return f"{nome} está com tudo em dia! Nenhum ponto de atenção foi identificado agora."
    top = " ".join(a["motivo"] for a in score["achados"][:2])
    return f"Atenção com {nome} (prioridade {score['nivel'].lower()}): {top} Recomendamos agendar um atendimento com o veterinário."

def fallback_resumo(ctx, score):
    p = ctx["pet"]; ult = ctx["prontuarios"][-1] if ctx["prontuarios"] else None
    linhas = [f"**Perfil**\n{p['nome']}, {p['especie']} {p['raca']} ({p['porte'].lower()}), {p['idade_anos']} anos.",
              "**O que mudou desde o último atendimento**\n" + (f"Último prontuário em {_br(_d(ult['data']))}: {ult['diagnostico']}." if ult else "Sem prontuários."),
              "**Pontos de atenção (motor de regras)**\n" + ("\n".join(f"- {a['motivo']}" for a in score["achados"]) or "Nenhum."),
              "**Tratamentos em curso**\n" + ("\n".join(f"- {r['tipo'].title()}: {r['descricao']}" for r in ctx["prescricoes"] if r.get("ativa", True)) or "Nenhum registrado.")]
    return "\n\n".join(linhas)

# ── Tarefas ──
def gerar_notificacao_tutor(ctx, score):
    try:
        return chamar_llm(SYSTEM_BASE, _prompt(TAREFA_NOTIFICACAO, ctx, score), max_tokens=200) + DISCLAIMER, "llm"
    except Exception as e:
        return fallback_notificacao(ctx, score) + DISCLAIMER, "regras (fallback)"

def gerar_resumo_veterinario(ctx):
    score = calcular_score(ctx)
    try:
        texto, origem = chamar_llm(SYSTEM_BASE, _prompt(TAREFA_RESUMO, ctx, score), max_tokens=500), "llm"
    except Exception:
        texto, origem = fallback_resumo(ctx, score), "regras (fallback)"
    return {"id_pet": ctx["id_pet"], "score": score["score"], "nivel": score["nivel"], "resumo": texto + DISCLAIMER, "origem_texto": origem}

def eh_emergencia(pergunta):
    q = _norm(pergunta)
    return any(re.search(p, q) for p in EMERGENCIAS)

def chat_tutor(ctx, pergunta):
    if eh_emergencia(pergunta):
        return {"id_pet": ctx["id_pet"], "resposta": ("⚠️ O que você descreveu pode ser uma **emergência**. Procure atendimento veterinário de urgência "
                "agora ou ligue para a clínica. Não dê medicamentos por conta própria."), "origem_texto": "regra_emergencia", "escalar_para_clinica": True}
    score = calcular_score(ctx)
    try:
        resp, origem = chamar_llm(SYSTEM_BASE, _prompt(TAREFA_CHAT, ctx, score, f"\nPERGUNTA DO TUTOR: {pergunta}"), max_tokens=300), "llm"
    except Exception:
        resp, origem = f"No momento não consegui consultar o assistente. Para essa dúvida sobre {ctx['pet']['nome']}, fale com o veterinário da clínica.", "regras (fallback)"
    return {"id_pet": ctx["id_pet"], "resposta": resp + DISCLAIMER, "origem_texto": origem, "escalar_para_clinica": False}

## 6. Orquestrador — contexto → score → LLM → notificação (DTO)

In [ ]:
NOTIFICACOES = []   # simula TB_NOTIFICACAO

def analisar_pet(ctx, gravar_notificacao=True):
    score = calcular_score(ctx)
    texto, origem = gerar_notificacao_tutor(ctx, score)
    dto = {"id_pet": ctx["id_pet"], "nome_pet": ctx["pet"]["nome"], "score": score["score"], "nivel": score["nivel"],
           "motivos": score["achados"], "mensagem_tutor": texto, "origem_texto": origem,
           "gerado_em": datetime.now().isoformat(timespec="seconds")}
    if gravar_notificacao and score["nivel"] != "BAIXO":
        NOTIFICACOES.append({"ID_PET": ctx["id_pet"], "MENSAGEM": texto, "PRIORIDADE": score["nivel"], "LIDA": False})
    return dto

def fila_clinica(contextos):
    # Rápida e sem LLM: só o motor de regras ordena a fila
    fila = []
    for c in contextos:
        s = calcular_score(c)
        fila.append({"id_pet": c["id_pet"], "pet": c["pet"]["nome"], "score": s["score"], "nivel": s["nivel"],
                     "principais_motivos": [a["motivo"] for a in s["achados"][:2]]})
    return sorted(fila, key=lambda x: -x["score"])

## 7. Testes e validação

In [ ]:
def ctx_base(**kw):
    c = {"id_pet": 99, "referencia_em": AGORA.isoformat(),
         "pet": {"nome": "Teste", "sexo": "M", "especie": "Cachorro", "raca": "SRD", "porte": "GRANDE", "idade_anos": 3},
         "prontuarios": [], "consultas": [], "prescricoes": [], "coleira_24h": [],
         "historico_saude": [{"evento": "VACINA", "data": passado(10), "detalhes": ""}]}
    c.update(kw); return c

def codigos(ctx): return {a["codigo"]: a["pontos"] for a in calcular_score(ctx)["achados"]}
def leituras(bpms, alerta=None): return [{"datahora": AGORA.isoformat(), "bpm": b, "alerta": alerta} for b in bpms]

def test_pet_saudavel_tem_score_zero():
    assert calcular_score(ctx_base())["score"] == 0

def test_vacina_atrasada_e_sem_registro():
    assert codigos(ctx_base(historico_saude=[{"evento": "VACINA", "data": passado(400), "detalhes": ""}])) == {"VACINA_ATRASADA": 20}
    assert codigos(ctx_base(historico_saude=[])) == {"VACINA_SEM_REGISTRO": 20}

def test_variacao_de_peso():
    p = [{"data": passado(60), "peso_kg": 10.0, "sintomas": ""}, {"data": passado(30), "peso_kg": 8.5, "sintomas": ""}]
    assert codigos(ctx_base(prontuarios=p)) == {"VARIACAO_PESO": 20}
    p[1]["peso_kg"] = 10.6   # +6% -> faixa intermediária
    assert codigos(ctx_base(prontuarios=p)) == {"VARIACAO_PESO": 10}

def test_retorno_atrasado_e_cumprido():
    p = [{"data": passado(30), "peso_kg": None, "sintomas": "", "data_retorno": passado(5)}]
    assert codigos(ctx_base(prontuarios=p)) == {"RETORNO_ATRASADO": 15}
    realizada = [{"data": passado(2), "status": "REALIZADA", "modalidade": "PRESENCIAL"}]
    assert codigos(ctx_base(prontuarios=p, consultas=realizada)) == {}

def test_bpm_isolado_vs_persistente_respeita_o_porte():
    assert codigos(ctx_base(coleira_24h=leituras([130]))) == {"BPM_ISOLADO": 10}          # grande: 60–100
    assert codigos(ctx_base(coleira_24h=leituras([130, 131, 132]))) == {"BPM_PERSISTENTE": 25}
    pequeno = ctx_base(coleira_24h=leituras([130, 131, 132])); pequeno["pet"]["porte"] = "PEQUENO"   # 100–140
    assert codigos(pequeno) == {}

def test_alerta_da_coleira():
    assert codigos(ctx_base(coleira_24h=leituras([80], alerta="TAQUICARDIA"))) == {"ALERTA_COLEIRA": 20}

def test_sintoma_recorrente():
    p = [{"data": passado(100), "sintomas": "Vômito", "peso_kg": None}, {"data": passado(10), "sintomas": "vômito, apatia", "peso_kg": None}]
    r = codigos(ctx_base(prontuarios=p))
    assert r == {"SINTOMAS_RECENTES": 10, "SINTOMA_RECORRENTE": 10}

def test_cenarios_do_seed_e_ordenacao_da_fila():
    esperado = {"Thor": ("BAIXO", 0), "Mel": ("MEDIO", 45), "Luna": ("ALTO", 55), "Rex": ("CRITICO", 80)}
    fila = fila_clinica(CONTEXTOS)
    for item in fila:
        assert (item["nivel"], item["score"]) == esperado[item["pet"]], item
    assert [i["pet"] for i in fila] == ["Rex", "Luna", "Mel", "Thor"]

def test_privacidade_nenhum_dado_pessoal_no_contexto():
    blob = json.dumps(CONTEXTOS, ensure_ascii=False).lower()
    assert "@" not in blob and "cpf" not in blob and "tutor exemplo" not in blob
    assert "latitude" not in blob and "longitude" not in blob and "(11)" not in blob
    # e nada disso vai para o prompt do LLM
    for c in CONTEXTOS:
        p = _prompt(TAREFA_NOTIFICACAO, c, calcular_score(c)).lower()
        assert "@" not in p and "cpf" not in p and "latitude" not in p

def test_emergencia_nao_chama_o_llm():
    global chamar_llm
    original = chamar_llm
    def explode(*a, **k): raise AssertionError("LLM não deveria ser chamado")
    chamar_llm = explode
    try:
        r = chat_tutor(CONTEXTOS[0], "Meu cachorro teve uma convulsão agora!")
        assert r["origem_texto"] == "regra_emergencia" and r["escalar_para_clinica"] is True
        assert eh_emergencia("ele comeu CHOCOLATE") and not eh_emergencia("qual a melhor hora para passear?")
    finally:
        chamar_llm = original

def test_fallback_quando_llm_esta_fora():
    global OLLAMA_URL
    original = OLLAMA_URL
    OLLAMA_URL = "http://127.0.0.1:9"   # porta sem serviço
    try:
        dto = analisar_pet(CONTEXTOS[3], gravar_notificacao=False)
        assert dto["origem_texto"] == "regras (fallback)" and dto["nivel"] == "CRITICO" and "Rex" in dto["mensagem_tutor"]
    finally:
        OLLAMA_URL = original

falhas = 0
for nome, fn in sorted((n, f) for n, f in globals().items() if n.startswith("test_") and callable(f)):
    try:
        fn(); print("✅", nome)
    except Exception as e:
        falhas += 1; print("❌", nome, "->", repr(e))
print(f"\n{'Todos os testes passaram.' if not falhas else f'{falhas} teste(s) falharam.'}")

## 8. Demonstração (use estas células no vídeo)
Fluxo mostrado: **clínica** vê a fila → **tutor** recebe a notificação → **veterinário** lê o resumo → **tutor** conversa com o assistente.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

fila = fila_clinica(CONTEXTOS)
display(pd.DataFrame([{**f, "principais_motivos": " | ".join(f["principais_motivos"])} for f in fila]))

In [ ]:
# TUTOR — notificação personalizada (regras decidem, LLM redige)
analise_rex = analisar_pet(CONTEXTOS[3])
print("Score:", analise_rex["score"], analise_rex["nivel"], "| texto:", analise_rex["origem_texto"], "\n")
for m in analise_rex["motivos"]: print(f"  +{m['pontos']:>2}  {m['codigo']}: {m['motivo']}")
display(Markdown("**Notificação para o tutor:**\n\n" + analise_rex["mensagem_tutor"]))

In [ ]:
# VETERINÁRIO — resumo pré-consulta
resumo_luna = gerar_resumo_veterinario(CONTEXTOS[2])
display(Markdown(f"### Resumo de Luna  ·  {resumo_luna['nivel']} ({resumo_luna['score']})  ·  texto: {resumo_luna['origem_texto']}\n\n" + resumo_luna["resumo"]))

In [ ]:
# TUTOR — assistente conversacional
chats = []
for idx, pergunta in [(1, "A Mel pode comer petisco hoje? Quando ela precisa voltar ao veterinário?"),
                      (0, "O Thor engoliu chocolate agora há pouco, o que eu faço?")]:
    r = chat_tutor(CONTEXTOS[idx], pergunta); chats.append({"pergunta": pergunta, **r})
    display(Markdown(f"**Tutor:** {pergunta}\n\n**SuperVet IA** _(origem: {r['origem_texto']})_:\n\n{r['resposta']}"))

## 9. Integração com o backend Java (Spring Boot)
O Colab expõe uma API REST **stateless**: o Java monta o JSON de contexto (mesmo formato da Seção 3, ele tem acesso ao Oracle), chama o Colab e grava o resultado em `TB_NOTIFICACAO`.

| Método | Rota | Corpo | Retorno |
|---|---|---|---|
| GET | `/health` | — | status do Ollama e modelo |
| GET | `/demo/contexto/{id_pet}` | — | exemplo do JSON que o Java deve enviar |
| POST | `/ai/prioridade` | `{"contexto": {...}}` | score, nível, motivos, mensagem do tutor |
| POST | `/ai/resumo-prontuario` | `{"contexto": {...}}` | resumo pré-consulta |
| POST | `/ai/chat` | `{"contexto": {...}, "pergunta": "..."}` | resposta do assistente |
| POST | `/ai/fila` | `{"contextos": [{...}]}` | pets ordenados por score (sem LLM) |

> A URL do túnel muda a cada execução do notebook. No Spring, leia-a de uma variável de ambiente (ex.: `SUPERVET_AI_URL`).

In [ ]:
!pip -q install fastapi "uvicorn[standard]" > /dev/null

In [ ]:
import threading, time, subprocess, re, uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="SuperVet IA", version="1.0")

class ContextoReq(BaseModel):
    contexto: dict

class ChatReq(BaseModel):
    contexto: dict
    pergunta: str

class FilaReq(BaseModel):
    contextos: list[dict]

def _valida(ctx):
    faltando = [k for k in ("id_pet", "referencia_em", "pet", "prontuarios", "consultas", "historico_saude", "prescricoes", "coleira_24h") if k not in ctx]
    if faltando:
        raise HTTPException(422, f"Contexto inválido, campos ausentes: {faltando}")

@app.get("/health")
def health():
    return {"status": "ok", "modelo": MODELO, "ollama_ativo": ollama_ativo()}

@app.get("/demo/contexto/{id_pet}")
def demo_contexto(id_pet: int):
    if id_pet not in TB_PET: raise HTTPException(404, "Pet não encontrado")
    return builder.montar(id_pet)

@app.post("/ai/prioridade")
def api_prioridade(req: ContextoReq):
    _valida(req.contexto)
    return analisar_pet(req.contexto, gravar_notificacao=False)   # quem grava TB_NOTIFICACAO é o Java

@app.post("/ai/resumo-prontuario")
def api_resumo(req: ContextoReq):
    _valida(req.contexto)
    return gerar_resumo_veterinario(req.contexto)

@app.post("/ai/chat")
def api_chat(req: ChatReq):
    _valida(req.contexto)
    return chat_tutor(req.contexto, req.pergunta)

@app.post("/ai/fila")
def api_fila(req: FilaReq):
    for c in req.contextos: _valida(c)
    return fila_clinica(req.contextos)

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
print(requests.get("http://127.0.0.1:8000/health").json())

In [ ]:
# Teste de ponta a ponta pela API local (simula a chamada que o Java fará)
ctx_rex = requests.get("http://127.0.0.1:8000/demo/contexto/4").json()
r = requests.post("http://127.0.0.1:8000/ai/prioridade", json={"contexto": ctx_rex}, timeout=300).json()
print(r["nivel"], r["score"], "|", r["origem_texto"])
print(r["mensagem_tutor"])

In [ ]:
# Túnel público (cloudflared, sem conta) para o Spring Boot conseguir chamar o Colab
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

tunel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL_PUBLICA, inicio = None, time.time()
while time.time() - inicio < 60 and not URL_PUBLICA:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunel.stdout.readline())
    if m: URL_PUBLICA = m.group(0)
print("URL pública:", URL_PUBLICA)
print("Teste:", f"curl {URL_PUBLICA}/health")

**Exemplo de chamada no Spring Boot (Java 21)** — para o README:
```java
RestClient ia = RestClient.create(System.getenv("SUPERVET_AI_URL"));

Map<String, Object> resposta = ia.post()
    .uri("/ai/prioridade")
    .contentType(MediaType.APPLICATION_JSON)
    .body(Map.of("contexto", contextoDoPet))   // montado a partir do Oracle, sem CPF/e-mail/GPS
    .retrieve()
    .body(new ParameterizedTypeReference<>() {});
// -> gravar resposta.get("mensagem_tutor") em TB_NOTIFICACAO
```

## 10. Evidências para o repositório e o vídeo

In [ ]:
evidencias = {
    "gerado_em": datetime.now().isoformat(timespec="seconds"),
    "modelo": MODELO,
    "fila_clinica": fila,
    "analise_tutor_rex": analise_rex,
    "resumo_veterinario_luna": resumo_luna,
    "chat_tutor": chats,
    "notificacoes_gravadas": NOTIFICACOES,
    "testes": "ver seção 7",
}
with open("evidencias_ia.json", "w", encoding="utf-8") as f:
    json.dump(evidencias, f, ensure_ascii=False, indent=2)
print("Salvo em evidencias_ia.json")
try:
    from google.colab import files
    files.download("evidencias_ia.json")
except Exception:
    pass